In [ ]:
pip install numpy pandas matplotlib scikit-learn torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

### Task 1 — A Single Neuron in NumPy

In [ ]:
# Task 1 — A Single Neuron in NumPy

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Initialize weights and bias
w = np.random.normal(0, 0.01, size=30)
b = np.random.normal(0, 0.01)

# Single neuron forward pass
def forward(x, w, b):
    z = np.dot(x, w) + b
    return sigmoid(z)

# Run on first 5 test rows
predicted_probabilities = forward(X_test_scaled[:5], w, b)

print("Predicted probabilities for first 5 test rows:")
print(predicted_probabilities)

in machine-learning terms, what model did you just implement?

In machine-learning terms, this is a single-neuron binary classification model, which is essentially logistic regression implemented with NumPy.

### Task 2 — A Two-Layer MLP in NumPy

In [ ]:
# Task 2 — A Two-Layer MLP in NumPy

class NumpyMLP:
    def __init__(self, input_size=30, hidden_size=8, output_size=1):
        # He initialization for first layer
        self.W1 = np.random.normal(
            0,
            np.sqrt(2 / input_size),
            size=(input_size, hidden_size)
        )
        self.b1 = np.zeros(hidden_size)

        # He initialization for output layer
        self.W2 = np.random.normal(
            0,
            np.sqrt(2 / hidden_size),
            size=(hidden_size, output_size)
        )
        self.b2 = np.zeros(output_size)

    def relu(self, z):
        return np.maximum(0, z)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        # Hidden layer
        self.z1 = np.dot(X, self.W1) + self.b1
        self.h1 = self.relu(self.z1)

        # Output layer
        self.z2 = np.dot(self.h1, self.W2) + self.b2
        output = self.sigmoid(self.z2)

        return output  


# Create the MLP
numpy_mlp = NumpyMLP(input_size=30, hidden_size=8, output_size=1)

# Run forward pass on test set
numpy_outputs = numpy_mlp.forward(X_test_scaled)

print("Output shape:")
print(numpy_outputs.shape)

print("\nFirst 5 predictions:")
print(numpy_outputs[:5])

Why does the output shape match what you'd want for a binary classifier even though we haven't trained anything yet?

The output shape is `(114, 1)` because the test set has 114 samples and the network returns one probability for each sample. This matches binary classification because each row needs one output value between 0 and 1, even though the weights are still random and the model is not trained yet.

### Task 3 — The Same Network in PyTorch

In [ ]:
# Task 3 — The Same Network in PyTorch

class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        z1 = self.fc1(x)
        h1 = torch.relu(z1)

        z2 = self.fc2(h1)
        output = torch.sigmoid(z2)

        return output


# Create PyTorch model
torch_mlp = TorchMLP()

# Copy NumPy weights into PyTorch model
torch_mlp.fc1.weight.data = torch.from_numpy(numpy_mlp.W1.T).float()
torch_mlp.fc1.bias.data = torch.from_numpy(numpy_mlp.b1).float()

torch_mlp.fc2.weight.data = torch.from_numpy(numpy_mlp.W2.T).float()
torch_mlp.fc2.bias.data = torch.from_numpy(numpy_mlp.b2).float()

# Convert test data from NumPy array to PyTorch tensor
X_test_tensor = torch.from_numpy(X_test_scaled).float()

# Run forward pass in PyTorch
torch_outputs = torch_mlp(X_test_tensor)

# Convert PyTorch output back to NumPy
torch_outputs_np = torch_outputs.detach().numpy()

# Compare NumPy and PyTorch outputs
max_difference = np.max(np.abs(numpy_outputs - torch_outputs_np))

print("First 5 NumPy predictions:")
print(numpy_outputs[:5])

print("\nFirst 5 PyTorch predictions:")
print(torch_outputs_np[:5])

print("\nMaximum absolute difference:")
print(max_difference)

The NumPy and PyTorch models produced almost identical predictions because the PyTorch model used the same architecture, same scaled test data, and the same copied weights from the NumPy model. The maximum absolute difference is only `1.379e-07`, which is extremely small and comes from tiny numerical precision differences.

### Task 4 — Activation Function Experiment

In [ ]:
# Task 4 — Activation Function Experiment

class ActivationMLP(nn.Module):
    def __init__(self, activation_name):
        super().__init__()

        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)
        self.activation_name = activation_name

    def hidden_activation(self, z):
        if self.activation_name == "sigmoid":
            return torch.sigmoid(z)
        elif self.activation_name == "tanh":
            return torch.tanh(z)
        elif self.activation_name == "relu":
            return torch.relu(z)

    def forward(self, x):
        z1 = self.fc1(x)
        h1 = self.hidden_activation(z1)
        z2 = self.fc2(h1)
        output = torch.sigmoid(z2)

        return output, z1, h1

In [ ]:
# Create three models with different hidden activations
activation_names = ["sigmoid", "tanh", "relu"]

pre_activations = {}
hidden_outputs = {}
predictions = {}

for name in activation_names:
    model = ActivationMLP(name)

    with torch.no_grad():
        output, z1, h1 = model(X_test_tensor)

    predictions[name] = output.numpy()
    pre_activations[name] = z1.numpy()
    hidden_outputs[name] = h1.numpy()

In [ ]:
# Plot hidden-layer pre-activations

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, activation_names):
    ax.hist(pre_activations[name].ravel(), bins=30)
    ax.set_title(f"{name.capitalize()} pre-activations")
    ax.set_xlabel("z values")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Plot hidden-layer outputs after activation

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, activation_names):
    ax.hist(hidden_outputs[name].ravel(), bins=30)
    ax.set_title(f"{name.capitalize()} hidden outputs")
    ax.set_xlabel("Activation values")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate saturation / inactive fractions

sigmoid_values = hidden_outputs["sigmoid"]
tanh_values = hidden_outputs["tanh"]
relu_values = hidden_outputs["relu"]

sigmoid_saturated = np.mean((sigmoid_values < 0.05) | (sigmoid_values > 0.95))
tanh_saturated = np.mean((tanh_values < -0.95) | (tanh_values > 0.95))
relu_inactive = np.mean(relu_values == 0)

print("Sigmoid saturated fraction:", sigmoid_saturated)
print("Tanh saturated fraction:", tanh_saturated)
print("ReLU inactive fraction:", relu_inactive)

**For Sigmoid and Tanh, what fraction of the post-activation values fall in the saturated region?**  
For sigmoid, the saturated fraction is about `0.0011`, which means almost none of the values are very close to 0 or 1. For tanh, the saturated fraction is about `0.0132`, so only a small number of values are close to -1 or 1.

**Looking at ReLU, roughly what fraction of the units are inactive?**  
For ReLU, the inactive fraction is about `0.5614`, meaning around 56% of the hidden outputs are exactly 0.

**Based on these histograms, why might ReLU be the better default for hidden layers?**  
ReLU can be a better default because it keeps positive values unchanged instead of squeezing everything into a small range like sigmoid or tanh. This can make learning easier in deeper networks, although in this random model many ReLU units are inactive because negative values become 0.